# Multi-Sample Integration: PBMC 3k + PBMC 4k

Integration of two PBMC datasets from different donors using Harmony for batch correction.
Demonstrates handling batch effects, joint clustering, and comparing cell type proportions across samples.

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

sc.settings.verbosity = 1
sc.settings.figdir = '../results/'
sc.settings.set_figure_params(dpi=100, frameon=False)

## Load and label samples

In [ ]:
adata_3k = sc.datasets.pbmc3k()
adata_3k.var_names_make_unique()
adata_3k.obs['sample'] = 'donor_1'

adata_4k = sc.read_10x_mtx('../data/filtered_gene_bc_matrices/GRCh38/')
adata_4k.var_names_make_unique()
adata_4k.obs['sample'] = 'donor_2'

print(f"Donor 1: {adata_3k.n_obs} cells")
print(f"Donor 2: {adata_4k.n_obs} cells")

## Merge and QC

In [ ]:
adata = sc.concat([adata_3k, adata_4k], join='inner')
print(f"Merged: {adata.n_obs} cells x {adata.n_vars} genes")

adata.var['mt'] = adata.var_names.str.startswith('MT-')
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

In [ ]:
sc.pl.violin(adata, ['n_genes_by_counts', 'total_counts', 'pct_counts_mt'],
             jitter=0.4, multi_panel=True, groupby='sample', save='_integration_qc.pdf')

In [ ]:
adata = adata[adata.obs.n_genes_by_counts < 2500, :].copy()
adata = adata[adata.obs.n_genes_by_counts > 200, :].copy()
adata = adata[adata.obs.pct_counts_mt < 5, :].copy()
sc.pp.filter_genes(adata, min_cells=3)

print(f"After QC: {adata.n_obs} cells x {adata.n_vars} genes")
print(adata.obs['sample'].value_counts())

## Normalize and find HVGs

In [ ]:
adata.raw = adata.copy()

sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
sc.pp.highly_variable_genes(adata, min_mean=0.0125, max_mean=3, min_disp=0.5, batch_key='sample')
print(f"{adata.var.highly_variable.sum()} HVGs selected")

In [ ]:
adata = adata[:, adata.var.highly_variable].copy()
sc.pp.regress_out(adata, ['total_counts', 'pct_counts_mt'])
sc.pp.scale(adata, max_value=10)

## PCA and UMAP without correction

First, look at the batch effect before integration.

In [ ]:
sc.tl.pca(adata, svd_solver='arpack')
sc.pp.neighbors(adata, n_neighbors=10, n_pcs=30)
sc.tl.umap(adata)

sc.pl.umap(adata, color='sample', save='_before_integration.pdf')

## Harmony integration

Correct for batch effects while preserving biological variation.

In [ ]:
import harmonypy

harmony_out = harmonypy.run_harmony(adata.obsm['X_pca'], adata.obs, 'sample')
adata.obsm['X_pca_harmony'] = harmony_out.Z_corr

In [ ]:
sc.pp.neighbors(adata, use_rep='X_pca_harmony', n_neighbors=10, n_pcs=30)
sc.tl.umap(adata)

sc.pl.umap(adata, color='sample', save='_after_integration.pdf')

## Clustering and annotation

In [ ]:
sc.tl.leiden(adata, resolution=0.8)
sc.pl.umap(adata, color='leiden', save='_integrated_clusters.pdf')

In [ ]:
sc.tl.rank_genes_groups(adata, 'leiden', method='wilcoxon')

marker_genes = {
    'CD4 T': ['IL7R', 'CD4'],
    'CD8 T': ['CD8A', 'CD8B'],
    'NK': ['GNLY', 'NKG7'],
    'B': ['MS4A1', 'CD79A'],
    'Monocytes': ['CD14', 'LYZ', 'FCGR3A'],
    'Dendritic': ['FCER1A', 'CST3'],
    'Platelets': ['PPBP']
}

sc.pl.dotplot(adata, var_names=marker_genes, groupby='leiden', save='_integrated_dotplot.pdf')

In [ ]:
# Print top markers to determine annotation
for i in sorted(adata.obs['leiden'].unique().tolist()):
    markers = sc.get.rank_genes_groups_df(adata, group=i).head(3)['names'].tolist()
    print(f"Cluster {i}: {markers}")

In [ ]:
# Annotate based on marker inspection
cluster_to_celltype = {
    '0': 'CD4 T Naive',
    '1': 'CD14 Monocytes',
    '2': 'CD4 T Memory',
    '3': 'B cells',
    '4': 'CD8 T',
    '5': 'CD8 T',
    '6': 'NK',
    '7': 'FCGR3A Monocytes',
    '8': 'Dendritic',
    '9': 'CD4 T Memory',
    '10': 'pDC',
    '11': 'Platelets'
}

adata.obs['cell_type'] = adata.obs['leiden'].map(cluster_to_celltype)
sc.pl.umap(adata, color='cell_type', save='_integrated_celltypes.pdf')

## Cell type proportions across donors

In [ ]:
prop = pd.crosstab(adata.obs['sample'], adata.obs['cell_type'], normalize='index')

fig, ax = plt.subplots(1, 1, figsize=(8, 4))
prop.plot(kind='bar', stacked=True, ax=ax)
ax.set_ylabel('Proportion')
ax.set_xlabel('')
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.savefig('../results/celltype_proportions.pdf')
plt.show()

## DE across donors within a cell type

In [ ]:
# Compare CD4 T cells between donors
cd4 = adata[adata.obs['cell_type'] == 'CD4 T'].copy()
sc.tl.rank_genes_groups(cd4, 'sample', method='wilcoxon')

de_df = sc.get.rank_genes_groups_df(cd4, group='donor_1')
de_df = de_df[de_df['pvals_adj'] < 0.05]
print(f"{len(de_df)} DE genes between donors in CD4 T cells")
de_df.head(10)

In [ ]:
adata.write('../results/pbmc_integrated.h5ad')
print('Done.')